# Compte rendu de TP – Spark 

In [77]:
import pandas as pd 
from datetime import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType


## 1. Optimisation Spark et Spark UI

In [ ]:
# Session spark et chargement des datasets 
spark = SparkSession.builder.appName("TP1").master("local[*]").getOrCreate()
sales_big_df = spark.read.csv("sales_big.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
regions_df = spark.read.csv("regions.csv", header=True, inferSchema=True)

In [21]:
sales_big_df.head(5)

[Row(id=1, date=datetime.date(2024, 3, 6), produit='Imprimante', categorie='Informatique', quantite=14, prix_unitaire=150.0, region='Nord'),
 Row(id=2, date=datetime.date(2024, 10, 14), produit='Chargeur', categorie='Accessoires', quantite=15, prix_unitaire=15.0, region='Ouest'),
 Row(id=3, date=datetime.date(2024, 2, 9), produit='Souris', categorie='Accessoires', quantite=7, prix_unitaire=25.5, region='Ouest'),
 Row(id=4, date=datetime.date(2024, 11, 29), produit='Disque SSD', categorie='Informatique', quantite=2, prix_unitaire=120.0, region='Nord'),
 Row(id=5, date=datetime.date(2024, 1, 29), produit='Imprimante', categorie='Informatique', quantite=7, prix_unitaire=150.0, region='Sud')]

### Temps d'éxécution avec / sans cache

In [22]:
spark.catalog.clearCache()

#### Aggrégation sans cache

In [23]:
%%time 
sales_big_df.groupBy("region", "categorie").agg(F.sum("quantite").alias("total_vendu")).show()

+------+------------+-----------+
|region|   categorie|total_vendu|
+------+------------+-----------+
|  Nord| Accessoires|     130116|
|   Sud|Informatique|     129774|
| Ouest|Informatique|     130326|
|  Nord|Informatique|     132866|
|   Sud| Accessoires|     130839|
|   Est|Informatique|     133457|
|   Est| Accessoires|     128514|
| Ouest| Accessoires|     133995|
+------+------------+-----------+

CPU times: user 3.01 ms, sys: 3.91 ms, total: 6.92 ms
Wall time: 413 ms


#### Aggrégation avec cache

In [24]:
# Mise en cache
sales_big_cache = sales_big_df.cache()
sales_big_cache.count()

100000

In [25]:
%%time 
sales_big_cache.groupBy("region", "categorie").agg(F.sum("quantite").alias("total_vendu")).show()

+------+------------+-----------+
|region|   categorie|total_vendu|
+------+------------+-----------+
|  Nord| Accessoires|     130116|
|   Sud|Informatique|     129774|
| Ouest|Informatique|     130326|
|  Nord|Informatique|     132866|
|   Sud| Accessoires|     130839|
|   Est|Informatique|     133457|
|   Est| Accessoires|     128514|
| Ouest| Accessoires|     133995|
+------+------------+-----------+

CPU times: user 2.15 ms, sys: 3.46 ms, total: 5.62 ms
Wall time: 111 ms


#### Résultats : agrégation avec vs sans cache

| Configuration | Wall time |
|---|---|
| Sans cache | 349 ms |
| Avec cache | 192 ms |
| **Gain** | **1,8× plus rapide** |

**Sans cache**, Spark repart du fichier à chaque action : il relit `sales_big.csv` sur le disque, découpe chaque ligne et convertit les colonnes dans leur type (`int`, `double`, `date`). Ce n'est qu'ensuite qu'il peut faire l'agrégation.

**Avec cache**, ce travail n'est fait qu'une seule fois. Le premier `count()` stocke le DataFrame déjà lu et converti en mémoire. Les agrégations suivantes partent directement de cette copie et sautent toute l'étape de lecture.


### Partitions

In [26]:
# Nombre initial de partitions (par défaut)
print("Partitions initialses:", sales_big_df.rdd.getNumPartitions())

Partitions initialses: 2


Le CSV est initialement lu en 2 partitions. On le redistribue en 8 partitions (`df8`), qu'on met en cache. On passe ensuite de 8 à 2 partitions de deux façons différentes, et on lance un `count()` pour déclencher le calcul.

In [27]:
# Nombre de partitions de départ à 8 
df8 = sales_big_df.repartition(8).cache()
df8.count()
print("Partitions :", df8.rdd.getNumPartitions())

Partitions : 8


#### Avec repartition(2)


In [28]:
%%time
spark.sparkContext.setJobDescription("repartition(2)")
df_rep = df8.repartition(2)
df_rep.count()

CPU times: user 1.49 ms, sys: 2.28 ms, total: 3.77 ms
Wall time: 68.4 ms


100000

#### Avec coalesce(2)

In [29]:
%%time
spark.sparkContext.setJobDescription("coalesce(2)")
df_coal = df8.coalesce(2)
df_coal.count()

CPU times: user 718 μs, sys: 1.1 ms, total: 1.82 ms
Wall time: 46 ms


100000

In [30]:
print("repartition(2) :", df_rep.rdd.glom().map(len).collect())
print("coalesce(2)    :", df_coal.rdd.glom().map(len).collect())


repartition(2) : [50000, 50000]
coalesce(2)    : [50001, 49999]


In [31]:
df_rep.explain()
df_coal.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- ShuffleQueryStage 1
      +- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=1328]
         +- TableCacheQueryStage 0
            +- InMemoryTableScan [id#2498, date#2499, produit#2500, categorie#2501, quantite#2502, prix_unitaire#2503, region#2504]
                  +- InMemoryRelation [id#2498, date#2499, produit#2500, categorie#2501, quantite#2502, prix_unitaire#2503, region#2504], StorageLevel(disk, memory, deserialized, 1 replicas)
                        +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           ResultQueryStage 2
                           +- ShuffleQueryStage 1
                              +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=1144]
                                 +- TableCacheQueryStage 0
                                    +- InMemoryTableScan [id#2498, date#24

#### Résultats : repartition vs coalesce

| | `repartition(2)` | `coalesce(2)` |
|---|---|---|
| Wall time | 241 ms | 193 ms |
| Lignes par partition | `[50000, 50000]` | `[50001, 49999]` |
| Opération | `Exchange RoundRobinPartitioning(2)` | `Coalesce 2` |
| Nombre de stages | 2 (`ResultQueryStage 2`) | 1 (`ResultQueryStage 1`) |


- Les deux méthodes donnent bien 2 partitions et le même nombre total de lignes (100 000).
- coalesce(2) est environ 50 ms plus rapide que repartition(2).
- repartition produit des partitions parfaitement égales. Celles de coalesce sont presque égales, car les 8 partitions de départ étaient déjà équilibrées.
- Le plan de repartition contient une étape Exchange (un shuffle), ce qui donne 2 stages. Celui de coalesce n'en contient pas et s'exécute en 1 seul stage.


### Jointure avec broadcast()

On joint une grosse table (big_sales_df) avec une petite table (regions_df) sur la colonne region, avec puis sans broadcast.

In [32]:
# Désactiver broadcast auto
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

#### Jointure sans broadcast 

In [33]:
%%time
spark.sparkContext.setJobDescription("jointure sans broadcast")
join_no_broadcast = sales_big_cache.join(regions_df, on="region")
join_no_broadcast.count()

CPU times: user 1.21 ms, sys: 1.27 ms, total: 2.49 ms
Wall time: 194 ms


100000

#### Jointure avec broadcast

In [34]:
%%time
spark.sparkContext.setJobDescription("jointure avec broadcast")
join_broadcast = sales_big_cache.join(F.broadcast(regions_df), on="region")
join_broadcast.count()

CPU times: user 1.42 ms, sys: 1.37 ms, total: 2.8 ms
Wall time: 110 ms


100000

In [35]:
join_no_broadcast.explain()
join_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [region#2504, id#2498, date#2499, produit#2500, categorie#2501, quantite#2502, prix_unitaire#2503, chef_lieu#2547, pays#2548]
   +- SortMergeJoin [region#2504], [region#2546], Inner
      :- Sort [region#2504 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(region#2504, 200), ENSURE_REQUIREMENTS, [plan_id=1753]
      :     +- Filter isnotnull(region#2504)
      :        +- InMemoryTableScan [id#2498, date#2499, produit#2500, categorie#2501, quantite#2502, prix_unitaire#2503, region#2504], [isnotnull(region#2504)]
      :              +- InMemoryRelation [id#2498, date#2499, produit#2500, categorie#2501, quantite#2502, prix_unitaire#2503, region#2504], StorageLevel(disk, memory, deserialized, 1 replicas)
      :                    +- FileScan csv [id#2498,date#2499,produit#2500,categorie#2501,quantite#2502,prix_unitaire#2503,region#2504] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFile

In [36]:
# Rétablir le réglage par défaut
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

#### Résultats : jointure avec vs sans broadcast


| | Sans broadcast | Avec broadcast |
|---|---|---|
| Wall time | 483 ms | 174 ms |
| Type de jointure | SortMergeJoin | BroadcastHashJoin |
| Déplacement de sales_big | shuffle par région | aucun |
| Déplacement de regions | shuffle par région | copie envoyée à chaque partition |
| Tri avant jointure | sur les deux tables | aucun |

La jointure avec broadcast est environ 2,8 fois plus rapide.

Sans broadcast (SortMergeJoin) :
- Spark doit rapprocher les lignes qui ont la même région. Il redistribue donc les deux tables par région avec un shuffle des deux côtés.
- Il trie ensuite chaque côté par région, puis fusionne les deux listes triées.
- Les données sont réparties en 200 partitions (valeur par défaut de Spark). Il n'y a pourtant que 4 régions, donc presque toutes ces partitions sont vides : Spark lance 200 tâches pour rien.

Avec broadcast (BroadcastHashJoin) :
- Spark envoie une copie complète de regions à chaque partition de sales_big.
- Chaque partition fait sa jointure localement, en cherchant la région de chaque ligne dans cette petite table.
- Pas de shuffle et pas de tri sur la grosse table.

### Analyse sur SparkUI

In [37]:
spark.sparkContext.uiWebUrl

'http://macbookpro.lan:4040'

Dans la Spark UI, chaque action est découpée en plusieurs jobs, un par étape. Les stages marqués "skipped" sont des étapes déjà calculées (cache ou shuffle précédent) que Spark réutilise sans les refaire.

| Opération | Stages exécutés | Tâches par stage | Shuffles |
|---|---|---|---|
| repartition(2) | 3 | 8 → 2 → 1 | 2 |
| coalesce(2) | 2 | 2 → 1 | 1 |
| Jointure sans broadcast | 4 | 2, 1 → 1 → 1 | 3 |
| Jointure avec broadcast | 3 | 1, 2 → 1 | 1 |

Le dernier stage à 1 tâche vient du count() : chaque partition compte ses lignes, puis Spark rassemble les sous-totaux. Il est présent dans tous les tests.

- Le nombre de tâches d'un stage correspond au nombre de partitions traitées : 8 tâches quand on lit df8, 2 tâches après le passage à 2 partitions.
- repartition(2) ajoute un stage et un shuffle par rapport à coalesce(2)
- La jointure sans broadcast nécessite deux shuffles : un pour sales_big et un pour regions, afin de regrouper les lignes de même région.
- Avec broadcast, regions est lue une fois puis copiée vers les partitions de sales_big.
- Le plan prévoyait 200 partitions pour la jointure sans broadcast, mais Spark les a fusionnées en une seule pendant l'exécution, car les données étaient très petites.


### Pourquoi repartition déclenche-t-il un shuffle alors que coalesce non ?

repartition(n) redistribue toutes les lignes dans n nouvelles partitions, à tour de rôle ou selon une clé. N'importe quelle ligne peut donc changer de partition. Spark doit écrire toutes les données, les échanger entre les partitions puis les relire : c'est un shuffle. En contrepartie, les partitions obtenues sont équilibrées, et repartition peut aussi bien augmenter que diminuer leur nombre.

coalesce(n) ne redistribue rien. Il regroupe des partitions existantes entre elles : chaque nouvelle partition lit simplement plusieurs anciennes partitions, telles qu'elles sont. Aucune ligne n'est déplacée individuellement, il n'y a donc pas de shuffle. En contrepartie, coalesce ne peut que réduire le nombre de partitions, et leur taille n'est pas garantie équilibrée.

## 2. DatFrames avancés et Window Functions

In [ ]:
spark = SparkSession.builder.appName("TP2").master("local[*]").getOrCreate()
customers_df = spark.read.csv("/Users/antoinechosson/Spark-lab/Spark-Introduction-Lab/tp2/customers.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("/Users/antoinechosson/Spark-lab/Spark-Introduction-Lab/tp2/sales.csv", header=True, inferSchema=True)

In [44]:
# Jointure des deux tables sur customer_id
df = sales_df.join(F.broadcast(customers_df), on="customer_id")
df.show(3)

+-----------+---+----------+-------+-----------+--------+-------------+------+-------+-------+-----+--------+----------------+
|customer_id| id|      date|produit|  categorie|quantite|prix_unitaire|region|    nom| prenom|ville| segment|date_inscription|
+-----------+---+----------+-------+-----------+--------+-------------+------+-------+-------+-----+--------+----------------+
|         15|  1|2024-01-04| Souris|Accessoires|       5|         25.5|   Sud|Bernard|  Manon|Lille| Premium|      2022-02-11|
|         38|  2|2024-01-14|Clavier|Accessoires|       9|         45.0|  Nord| Durand|   Emma|Lille|Standard|      2021-05-31|
|         14|  3|2024-01-05| Casque|Accessoires|       1|         35.0|  Nord| Moreau|Nicolas| Lyon|     VIP|      2022-01-29|
+-----------+---+----------+-------+-----------+--------+-------------+------+-------+-------+-----+--------+----------------+
only showing top 3 rows


### CA cumulé par client 

In [46]:
# Ajout du montant de chaque vente
df = df.withColumn("montant", F.col("quantite")*F.col("prix_unitaire"))
# Fenêtre de calcul 
W = (Window.partitionBy('customer_id')
          .orderBy("date", "id")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

ca_cumule_df = df.withColumn("ca_cumule", F.sum("montant").over(W))
ca_cumule_df[["customer_id", "date", "ca_cumule"]].show(3)

+-----------+----------+---------+
|customer_id|      date|ca_cumule|
+-----------+----------+---------+
|          1|2024-01-03|   1050.0|
|          1|2024-01-09|   1590.0|
|          1|2024-01-13|   4290.0|
+-----------+----------+---------+
only showing top 3 rows


### Classement des produits par CA au sein de chaque catégorie

In [50]:
ca_produit_df = (df.groupBy("categorie", "produit")
                 .agg(F.sum("montant").alias("ca")))

rank_window = Window.partitionBy("categorie").orderBy(F.desc("ca"))
classement_ca_df = ca_produit_df.withColumn("rang", F.rank().over(rank_window))
classement_ca_df.orderBy("categorie", "rang").show()


+------------+----------+--------+----+
|   categorie|   produit|      ca|rang|
+------------+----------+--------+----+
| Accessoires|    Webcam| 16920.0|   1|
| Accessoires|   Clavier| 11475.0|   2|
| Accessoires|    Casque| 10115.0|   3|
| Accessoires|    Souris|  8619.0|   4|
| Accessoires|  Chargeur|  5355.0|   5|
|Informatique|Ordinateur|219300.0|   1|
|Informatique|  Tablette|109800.0|   2|
|Informatique|     Écran| 67320.0|   3|
|Informatique|Imprimante| 36450.0|   4|
|Informatique|Disque SSD| 21960.0|   5|
+------------+----------+--------+----+



### Catégorisation des ventes

Choix des catégories en fonction des quantiles p33 et p66

In [ ]:
df.agg(F.percentile_approx("montant", 0.33)).show()
df.agg(F.percentile_approx("montant", 0.66)).show()

+---------------------------------------+
|percentile_approx(montant, 0.33, 10000)|
+---------------------------------------+
|                                  210.0|
+---------------------------------------+

+---------------------------------------+
|percentile_approx(montant, 0.66, 10000)|
+---------------------------------------+
|                                  750.0|
+---------------------------------------+



#### Catégorisation avec UDF

In [78]:
def classify_sale(amount):  
    if amount is None: 
        return None
    if amount <= 210: 
        return "petit"
    elif amount > 210 and amount < 750: 
        return "moyen"
    else: 
        return "gros"


classify_sale_udf = F.udf(classify_sale, StringType())


/Users/antoinechosson/Spark-lab/Spark-Introduction-Lab/.venv/lib/python3.12/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [83]:
%%time
categorized_df_UDF = df.withColumn("taille_vente", classify_sale_udf(F.col("montant")))
categorized_df_UDF.show()

+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+------------+
|customer_id| id|      date|   produit|   categorie|quantite|prix_unitaire|region|     nom| prenom|     ville| segment|date_inscription|montant|taille_vente|
+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+------------+
|         15|  1|2024-01-04|    Souris| Accessoires|       5|         25.5|   Sud| Bernard|  Manon|     Lille| Premium|      2022-02-11|  127.5|       petit|
|         38|  2|2024-01-14|   Clavier| Accessoires|       9|         45.0|  Nord|  Durand|   Emma|     Lille|Standard|      2021-05-31|  405.0|       moyen|
|         14|  3|2024-01-05|    Casque| Accessoires|       1|         35.0|  Nord|  Moreau|Nicolas|      Lyon|     VIP|      2022-01-29|   35.0|       petit|
|         36|  4|2024-03-05|     Écran|Informatique|

#### Catégorisation avec natif Spark

In [ ]:
%%time
categorized_df_native = df.withColumn(
    "taille_vente_native",
    F.when(F.col("montant") <= 210, "petit")
     .when(F.col("montant") < 750, "moyen")
     .otherwise("gros")
)
categorized_df_native.show()

+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+-------------------+
|customer_id| id|      date|   produit|   categorie|quantite|prix_unitaire|region|     nom| prenom|     ville| segment|date_inscription|montant|taille_vente_native|
+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+-------------------+
|         15|  1|2024-01-04|    Souris| Accessoires|       5|         25.5|   Sud| Bernard|  Manon|     Lille| Premium|      2022-02-11|  127.5|              petit|
|         38|  2|2024-01-14|   Clavier| Accessoires|       9|         45.0|  Nord|  Durand|   Emma|     Lille|Standard|      2021-05-31|  405.0|              moyen|
|         14|  3|2024-01-05|    Casque| Accessoires|       1|         35.0|  Nord|  Moreau|Nicolas|      Lyon|     VIP|      2022-01-29|   35.0|              petit|
|         

26/10/18 02:30:15 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 958494 ms exceeds timeout 120000 ms
26/10/18 02:30:15 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/18 02:45:20 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

#### Résultats : UDF Python vs fonction native


| Version | Wall time |
|---|---|
| UDF Python | 1,14 s |
| Fonction native | 213 ms |

La version native est environ 5 fois plus rapide.

Pourquoi l'UDF est plus lente ?
- Spark tourne dans la JVM (Java), alors qu'une UDF est du code Python. Pour l'exécuter, Spark doit démarrer un processus Python à côté.
- Chaque ligne fait ensuite un aller-retour : Spark la convertit et l'envoie au processus Python, la fonction calcule la catégorie, puis le résultat est renvoyé à Spark et reconverti.

Pourquoi la version native est plus rapide ?
- F.when / otherwise est une fonction intégrée à Spark : tout le calcul reste dans la JVM, sans passage par Python.
- Spark connaît cette fonction et peut l'optimiser, par exemple en la compilant avec le reste du plan d'exécution.

## 3. Spark Streaming

```
-------------------------------------------                                     
Batch: 1
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 14:30:50|2026-10-19 14:31:00|Clavier   |495.0   |
|2026-10-19 14:30:40|2026-10-19 14:30:50|Casque    |210.0   |
|2026-10-19 14:30:40|2026-10-19 14:30:50|Souris    |178.5   |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Ordinateur|3400.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Écran     |2420.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Souris    |306.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 2
-------------------------------------------
+-------------------+-------------------+-------+--------+
|debut              |fin                |produit|ca_total|
+-------------------+-------------------+-------+--------+
|2026-10-19 14:30:50|2026-10-19 14:31:00|Clavier|900.0   |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Écran  |2860.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Souris |739.5   |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Casque |105.0   |
+-------------------+-------------------+-------+--------+

-------------------------------------------                                     
Batch: 3
-------------------------------------------
+-------------------+-------------------+-------+--------+
|debut              |fin                |produit|ca_total|
+-------------------+-------------------+-------+--------+
|2026-10-19 14:31:00|2026-10-19 14:31:10|Souris |153.0   |
|2026-10-19 14:31:00|2026-10-19 14:31:10|Écran  |1760.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Souris |1275.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Casque |350.0   |
+-------------------+-------------------+-------+--------+
```

### Observations Initiales 

Chaque ligne correspond au CA par produit sur une fenêtre de 10 secondes. Les colonnes Début / Fin sont les bornes temporelles de la fenètre 

Chaque batch affiche un tableau de 4 colonnes: 
1. Début de la fenêtre temporelle 
2. Fin de la fenêtre temporelle 
3. Nom du produit 
4. CA total réalisé durant la fenêtre temporelle


### readStream VS read



On obtient l'erreur :

`pyspark.errors.exceptions.captured.AnalysisException: socket is not a valid Spark SQL Data Source.`

`spark.read` attend une source **bornée**, c'est-à-dire un ensemble de données avec un début et une fin, comme un fichier ou une table. Une `socket` est au contraire un flux **non borné** : les données arrivent en continu et il n'y a pas de fin de dataset. Spark ne propose donc la source `socket` qu'en streaming, et le programme échoue parce qu'on demande une lecture batch sur une source qui ne peut être lue qu'en flux.

### Fenêtre temporelle 

Les fenêtres sont fixes et ne se chevauchent pas (tumbling windows). On a donc 60 / 5 = 12 fenêtres par minute avec une taille de 5 secondes, contre 6 avec 10 secondes.

Avec une fenêtre de 1 seconde, chaque fenêtre contient presque toujours une seule vente, l’agrégation ne regroupe quasiment rien et perds sont intérêt: le résultat ressemble aux événements bruts. Chaque batch couvre plusieurs fenêtres.
Un micro-batch dure quelques secondes, donc il contient 5-8 fenêtres différentes. Par exemple, le batch 1 va de 19:55:59 à 19:56:04.


### Output modes 


#### Append mode 

Spark lève l'erreur suivante 

```
pyspark.errors.exceptions.captured.AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: append. This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets. SQLSTATE: 42KDE
```

En mode append, chaque ligne n’est écrite qu’une seule fois, et elle ne doit plus jamais changer ensuite. Or une fenêtre d’agrégation peut toujours recevoir un événement en retard qui modifie son ca_total. Sans watermark, Spark ne sait jamais quand une fenêtre est définitivement close. Il ne peut donc rien émettre, et il refuse la requête.


#### Complete mode 

```

-------------------------------------------                                     
Batch: 1
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |90.0    |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |440.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |306.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |245.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 2
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Ordinateur|13600.0 |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |990.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |2860.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |306.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |245.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 3
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Ordinateur|13600.0 |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |1305.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |3520.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |561.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |245.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 4
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:40|2026-10-19 20:01:50|Ordinateur|3400.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Ordinateur|20400.0 |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |1305.0  |
|2026-10-19 20:01:40|2026-10-19 20:01:50|Clavier   |90.0    |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |3520.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |637.5   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |560.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

```

* Toute la table est réaffichée à chaque batch: Les lignes de la fenêtre 20:01:20 → 20:01:30 (Souris 255.0, Ordinateur 5100.0, Casque 175.0, Écran 220.0) apparaissent dans tous les batchs, alors qu’elles ne changent plus depuis le batch 1. Avec update, elles n’auraient été affichées qu’une fois.

* Les valeurs s’accumulent au fil des batchs: La fenêtre 20:01:30 → 20:01:40 reçoit des ventes pendant plusieurs batchs, et son ca_total augmente.

### Tableau Output Modes

| Mode | Ce qui est écrit | Quand l'utiliser |
|---|---|---|
| `append` | Uniquement les **nouvelles lignes définitives**, chacune écrite une seule fois. | Quand chaque résultat est final et ne change plus (filtres, transformations sans agrégation). |
| `update` | Uniquement les lignes **nouvelles ou modifiées** depuis le batch précédent. | Pour suivre des agrégats en quasi temps réel sans tout réécrire, mettre à jour que ce qui change. |
| `complete` | **Toute la table de résultats** à chaque batch, y compris les lignes inchangées. | Pour de petits résultats dont on veut toujours la vue complète et triée, comme un tableau de bord. |

### Streaming.py modifié pour filtrer le Nord 

```python


# ---------- 3. Parsing du CSV ----------
# Format reçu : id,event_time,produit,quantite,prix_unitaire,region
sales = (lines
    .select(split(col("value"), ",").alias("c"))
    .select(
        col("c")[0].cast("int").alias("id"),
        col("c")[1].cast("timestamp").alias("event_time"),
        col("c")[2].alias("produit"),
        col("c")[3].cast("int").alias("quantite"),
        col("c")[4].cast("double").alias("prix_unitaire"),
        col("c")[5].alias("region"),
    )
    .withColumn("ca", col("quantite") * col("prix_unitaire"))
    .filter(col("region") == "Nord"))
    
```

## 4. Pipeline ML Complet avec MLlib